# Reimplementation of GCN using Message Passing
According to [Semi-Supervised Classification with Graph Convolutional Networks](https://arxiv.org/abs/1609.02907) by Thomas N. Kipf, Max Welling

In [24]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree

### Initialization
Takes the number of input (in_channels) and output channels (out_channels)
Calls the parent class constructor with aggregation method 'add'.
Defines a linear transformation W to be applied to the node features (part of the propagation rule)

### Forward
Defines the forward pass of the GCN layer </br>
__Self-loops__: Adds self-loops to the adjacency matrix. Self-loops are added to ensure that each node includes its own feature in the aggregation (aligns with the modified adjacency matrix Ā = A + I).

__Row and Col__: Extracts the row and column indices from edge_index. </br>
__Degree__: Computes the degree of each node, which is the number of connections a node has. </br>
__Degree Inverse Square Root__: Takes the inverse square root of the degree. D<sup>-0.5</sup> </br>
__Normalization Coefficient__: Computes the normalization factor D^<sup>-0.5</sup>A^D^<sup>-0.5</sup> </br>
__Linear Transformation__: Applies the linear transformation W to the node features X
__Message Propagation__: Calls the propagate method to start message passing. The propagate method uses the edge_index to pass messages along edges and x as the node features. The normalization factors norm are passed as additional information.

__Message Method__: Defines how messages are created.
- Normalization: Multiplies the features of the neighboring nodes x<sub>j</sub> by the normalization factor norm. This corresponds to the normalized adjacency matrix application.
__Update Method__: Defines how the aggregated messages are used to update the node features.
Return Aggregation Output: Simply returns the aggregated messages as the updated node features H<sup>l+1</sup>

In [25]:
class GCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin= torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        x = self.lin(x)

        return self.propagate(edge_index, x=x, norm=norm)

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j

    def update(self, aggr_out):
        return aggr_out

In [26]:
class GCN(torch.nn.Module):
    def __init__(self, num_features, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, num_classes)
        
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.dropout(x, p=0.5, training=self.training)
        return F.log_softmax(x, dim=1)

In [27]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='/tmp/Cora', name='Cora')

In [28]:
model = GCN(num_features=dataset.num_features, hidden_channels=16, num_classes=dataset.num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

In [29]:
model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(dataset.x, dataset.edge_index)
    train_loss = criterion(out[dataset.train_mask], dataset.y[dataset.train_mask])
    val_loss = criterion(out[dataset.val_mask], dataset.y[dataset.val_mask])
    pred = out.argmax(dim=1)
    train_acc = pred[dataset.train_mask].eq(dataset.y[dataset.train_mask]).sum().item() / dataset.train_mask.sum().item()
    val_acc = pred[dataset.val_mask].eq(dataset.y[dataset.val_mask]).sum().item() / dataset.val_mask.sum().item()
    print(f'Epoch {epoch+1}/{200}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
    train_loss.backward()
    optimizer.step()

Epoch 1/200, Train Loss: 1.9691, Train Acc: 0.1714, Val Loss: 1.9096, Val Acc: 0.1840
Epoch 2/200, Train Loss: 1.9384, Train Acc: 0.2071, Val Loss: 1.9124, Val Acc: 0.1960
Epoch 3/200, Train Loss: 1.8709, Train Acc: 0.2643, Val Loss: 1.8808, Val Acc: 0.2500
Epoch 4/200, Train Loss: 1.8012, Train Acc: 0.3643, Val Loss: 1.8864, Val Acc: 0.2260
Epoch 5/200, Train Loss: 1.7192, Train Acc: 0.4571, Val Loss: 1.8549, Val Acc: 0.3000
Epoch 6/200, Train Loss: 1.7219, Train Acc: 0.3571, Val Loss: 1.8506, Val Acc: 0.2760
Epoch 7/200, Train Loss: 1.6421, Train Acc: 0.4143, Val Loss: 1.8230, Val Acc: 0.2940
Epoch 8/200, Train Loss: 1.6545, Train Acc: 0.3714, Val Loss: 1.7875, Val Acc: 0.2980
Epoch 9/200, Train Loss: 1.5374, Train Acc: 0.4214, Val Loss: 1.7617, Val Acc: 0.3080
Epoch 10/200, Train Loss: 1.4909, Train Acc: 0.4714, Val Loss: 1.7211, Val Acc: 0.3540
Epoch 11/200, Train Loss: 1.3651, Train Acc: 0.5286, Val Loss: 1.7394, Val Acc: 0.3380
Epoch 12/200, Train Loss: 1.3644, Train Acc: 0.4929,

In [30]:
model.eval()
with torch.no_grad():
    logits = model(dataset.x, dataset.edge_index)
    pred = logits.argmax(dim=1)
    test_acc = pred[dataset.test_mask].eq(dataset.y[dataset.test_mask]).sum().item() / dataset.test_mask.sum().item()
    print(f'Test Accuracy: {test_acc}')

Test Accuracy: 0.792
